In [10]:
# Script: GenerarCsvBruto_Completo.py
import os
import xml.etree.ElementTree as ET
import pandas as pd
from tqdm import tqdm



def generar_csv_completo(CARPETA_TRAIN = "brain-1h-over-375day",ARCHIVO_SALIDA = "brain_bruto_1h.csv"):
    if not os.path.exists(CARPETA_TRAIN):
        print(f"❌ La carpeta '{CARPETA_TRAIN}' no existe.")
        return

    ficheros = sorted([f for f in os.listdir(CARPETA_TRAIN) if f.endswith(".xml")])
    print(f"📂 Encontrados {len(ficheros)} archivos XML. Iniciando proceso...")

    # Namespace del XML detectado anteriormente
    ns = {'ns': 'http://sndlib.zib.de/network'}
    
    datos_temporales = []

    # Bucle principal con barra de progreso
    for f in tqdm(ficheros, desc="Procesando XMLs"):
        ruta_completa = os.path.join(CARPETA_TRAIN, f)
        try:
            tree = ET.parse(ruta_completa)
            root = tree.getroot()
            
            # 1. Extraer Tiempo (tag <time> dentro de <meta>)
            tag_time = root.find(".//ns:time", ns)
            ts = tag_time.text if tag_time is not None else f.replace(".xml", "")
            
            fila = {'timestamp': ts}
            
            # 2. Extraer todas las demandas individuales
            for demand in root.findall(".//ns:demand", ns):
                src = demand.find("ns:source", ns).text
                dst = demand.find("ns:target", ns).text
                val_elem = demand.find("ns:demandValue", ns)
                
                if val_elem is not None:
                    nombre_enlace = f"{src}_{dst}"
                    fila[nombre_enlace] = float(val_elem.text)
            
            datos_temporales.append(fila)
            
        except Exception as e:
            # Si un archivo falla, continuamos con el siguiente
            continue

    print(f"\n📊 Creando DataFrame gigante (esto consumirá RAM)...")
    df = pd.DataFrame(datos_temporales)
    
    # 3. Limpieza y Ordenación
    print("🧹 Ordenando datos por tiempo...")
    df['timestamp'] = pd.to_datetime(df['timestamp'], format='%Y%m%d-%H%M', errors='coerce')
    df = df.sort_values('timestamp')
    
    # Reordenar columnas para que timestamp sea la primera
    cols = ['timestamp'] + sorted([c for c in df.columns if c != 'timestamp'])
    df = df[cols]
    
    print(f"💾 Guardando archivo CSV: {ARCHIVO_SALIDA}...")
    # Usamos index=False para no añadir la columna de IDs de pandas
    df.to_csv(ARCHIVO_SALIDA, index=False)
    
    print("-" * 40)
    print(f"✅ ¡PROCESO FINALIZADO!")
    print(f"Filas (instantes de tiempo): {len(df)}")
    print(f"Columnas (enlaces): {len(df.columns) - 1}")
    print(f"Archivo generado: {ARCHIVO_SALIDA}")
    print("-" * 40)

if __name__ == "__main__":
    generar_csv_completo(CARPETA_TRAIN = "brain-1h-over-375day",ARCHIVO_SALIDA = "brain_bruto_1h.csv")
    generar_csv_completo(CARPETA_TRAIN = "brain-1min-over-7days",ARCHIVO_SALIDA = "brain_bruto_1min.csv")

📂 Encontrados 8993 archivos XML. Iniciando proceso...


Procesando XMLs: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 8993/8993 [17:43<00:00,  8.45it/s]



📊 Creando DataFrame gigante (esto consumirá RAM)...
🧹 Ordenando datos por tiempo...
💾 Guardando archivo CSV: brain_bruto_1h.csv...
----------------------------------------
✅ ¡PROCESO FINALIZADO!
Filas (instantes de tiempo): 8993
Columnas (enlaces): 17663
Archivo generado: brain_bruto_1h.csv
----------------------------------------
📂 Encontrados 9723 archivos XML. Iniciando proceso...


Procesando XMLs: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 9723/9723 [19:59<00:00,  8.10it/s]



📊 Creando DataFrame gigante (esto consumirá RAM)...
🧹 Ordenando datos por tiempo...
💾 Guardando archivo CSV: brain_bruto_1min.csv...
----------------------------------------
✅ ¡PROCESO FINALIZADO!
Filas (instantes de tiempo): 9723
Columnas (enlaces): 14311
Archivo generado: brain_bruto_1min.csv
----------------------------------------


In [11]:
import pandas as pd
import os

def agrupar_por_nodos(archivo_entrada, archivo_salida):
    if not os.path.exists(archivo_entrada):
        print(f"❌ No existe el archivo: {archivo_entrada}")
        return

    print(f"📖 Cargando cabecera de {archivo_entrada}...")
    # Leemos solo la primera fila para crear el mapa de nombres
    df_sample = pd.read_csv(archivo_entrada, nrows=1)
    todas_las_columnas = [c for c in df_sample.columns if c != 'timestamp']
    
    # Mapa: 'HU4_ADH63' -> 'HU_ADH'
    mapa_agrupacion = {}
    for col in todas_las_columnas:
        partes = col.split('_')
        if len(partes) == 2:
            src = ''.join([i for i in partes[0] if not i.isdigit()])
            dst = ''.join([i for i in partes[1] if not i.isdigit()])
            mapa_agrupacion[col] = f"{src}_{dst}"

    print(f"🧩 Mapeo completado. Enlaces core detectados: {len(set(mapa_agrupacion.values()))}")

    # Procesar por trozos para no reventar la RAM
    print("🚀 Iniciando agregación...")
    chunks = pd.read_csv(archivo_entrada, chunksize=1000)
    lista_df_agrupados = []

    for i, chunk in enumerate(chunks):
        ts = chunk['timestamp'].reset_index(drop=True)
        chunk_datos = chunk.drop(columns=['timestamp'])
        
        # --- SOLUCIÓN AL ERROR DE AXIS ---
        # Agrupamos usando el mapa sobre las columnas
        # T (transponemos), agrupamos por el mapa, sumamos y volvemos a transponer
        chunk_agrupado = chunk_datos.T.groupby(mapa_agrupacion).sum().T.reset_index(drop=True)
        
        # Volvemos a poner el timestamp
        chunk_agrupado.insert(0, 'timestamp', ts)
        lista_df_agrupados.append(chunk_agrupado)
        print(f"   Bloque {i+1} procesado...")

    print("🔗 Uniendo bloques...")
    df_final = pd.concat(lista_df_agrupados, ignore_index=True)

    print(f"💾 Guardando en {archivo_salida}...")
    df_final.to_csv(archivo_salida, index=False)
    
    print("-" * 30)
    print(f"✅ ¡TERMINADO!")
    print(f"Archivo: {archivo_salida}")
    print(f"Columnas finales: {len(df_final.columns)}")
    print("-" * 30)

if __name__ == "__main__":
    # Ejecutar para el de 1 hora
    agrupar_por_nodos("brain_bruto_1h.csv", "brain_CORE_1h.csv")
    agrupar_por_nodos("brain_bruto_1min.csv", "brain_CORE_1min.csv")

📖 Cargando cabecera de brain_bruto_1h.csv...
🧩 Mapeo completado. Enlaces core detectados: 81
🚀 Iniciando agregación...
   Bloque 1 procesado...
   Bloque 2 procesado...
   Bloque 3 procesado...
   Bloque 4 procesado...
   Bloque 5 procesado...
   Bloque 6 procesado...
   Bloque 7 procesado...
   Bloque 8 procesado...
   Bloque 9 procesado...
🔗 Uniendo bloques...
💾 Guardando en brain_CORE_1h.csv...
------------------------------
✅ ¡TERMINADO!
Archivo: brain_CORE_1h.csv
Columnas finales: 82
------------------------------
📖 Cargando cabecera de brain_bruto_1min.csv...
🧩 Mapeo completado. Enlaces core detectados: 81
🚀 Iniciando agregación...
   Bloque 1 procesado...
   Bloque 2 procesado...
   Bloque 3 procesado...
   Bloque 4 procesado...
   Bloque 5 procesado...
   Bloque 6 procesado...
   Bloque 7 procesado...
   Bloque 8 procesado...
   Bloque 9 procesado...
   Bloque 10 procesado...
🔗 Uniendo bloques...
💾 Guardando en brain_CORE_1min.csv...
------------------------------
✅ ¡TERMINADO!

In [13]:
import pandas as pd
import numpy as np

def analizar_archivo(ruta_csv):
    print(f"\n🔍 Analizando: {ruta_csv}")
    print("-" * 40)
    
    try:
        # Cargamos el archivo
        df = pd.read_csv(ruta_csv)
        
        # 1. Conteo total de NaNs
        total_nans = df.isna().sum().sum()
        
        # 2. Filas totalmente vacías
        filas_vacias = df.drop(columns=['timestamp']).isna().all(axis=1).sum()
        
        # 3. Columnas con NaNs
        columnas_con_nans = df.columns[df.isna().any()].tolist()
        
        print(f"✅ Total de celdas en el archivo: {df.size}")
        print(f"⚠️ Total de NaNs encontrados: {total_nans}")
        print(f"🚫 Filas completamente vacías: {filas_vacias}")
        print(f"📊 Número de columnas: {len(df.columns)}")
        print(f"📈 Número de filas: {len(df)}")

        if total_nans > 0:
            print("\n📍 Detalle de NaNs por columna (Top 10):")
            detalles = df[columnas_con_nans].isna().sum().sort_values(ascending=False).head(10)
            print(detalles)
            
            # Opción para limpiar en el momento
            print("\n💡 Sugerencia: Usa 'df.fillna(0)' si quieres convertir los NaNs en tráfico cero.")
        else:
            print("\n💎 ¡Perfecto! El archivo no tiene valores nulos.")

    except Exception as e:
        print(f"❌ Error al leer el archivo: {e}")

if __name__ == "__main__":
    # Comprueba el que acabamos de agrupar
    analizar_archivo("brain_CORE_1h.csv")
    analizar_archivo("brain_CORE_1min.csv")
    # También puedes comprobar el bruto si quieres
    # analizar_archivo("brain_bruto_1h.csv")


🔍 Analizando: brain_CORE_1h.csv
----------------------------------------
✅ Total de celdas en el archivo: 737426
⚠️ Total de NaNs encontrados: 0
🚫 Filas completamente vacías: 0
📊 Número de columnas: 82
📈 Número de filas: 8993

💎 ¡Perfecto! El archivo no tiene valores nulos.

🔍 Analizando: brain_CORE_1min.csv
----------------------------------------
✅ Total de celdas en el archivo: 797286
⚠️ Total de NaNs encontrados: 0
🚫 Filas completamente vacías: 0
📊 Número de columnas: 82
📈 Número de filas: 9723

💎 ¡Perfecto! El archivo no tiene valores nulos.


In [16]:
import pandas as pd
import numpy as np

def enriquecer_y_guardar(archivo_entrada, archivo_salida):
    print(f"📖 Leyendo {archivo_entrada}...")
    df = pd.read_csv(archivo_entrada)
    
    # Asegurar formato fecha
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    
    print("🛠️ Añadiendo características de calendario...")
    
    # 1. Componentes básicos
    df['hour'] = df['timestamp'].dt.hour
    df['day_of_week'] = df['timestamp'].dt.dayofweek # Lunes=0, Domingo=6
    
    # 2. Codificación Cíclica (Seno/Coseno)
    # Esto ayuda al modelo a entender que las 23:00 y las 00:00 están pegadas
    df['hour_sin'] = np.sin(2 * np.pi * df['hour']/24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour']/24)
    
    # Esto ayuda a entender que el Domingo está pegado al Lunes
    df['day_sin'] = np.sin(2 * np.pi * df['day_of_week']/7)
    df['day_cos'] = np.cos(2 * np.pi * df['day_of_week']/7)
    
    # 3. Variable binaria de fin de semana
    df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
    
    # 4. Limpieza: Rellenar posibles huecos con 0 (muy importante)
    df = df.fillna(0)
    
    print(f"💾 Guardando archivo final: {archivo_salida}...")
    df.to_csv(archivo_salida, index=False)
    
    print("-" * 30)
    print("✅ ¡ARCHIVO FINAL LISTO!")
    print(f"Nuevas columnas añadidas: hour_sin, hour_cos, day_sin, day_cos, is_weekend")
    print(f"Total de columnas: {len(df.columns)}")
    print("-" * 30)

if __name__ == "__main__":
    # Generamos los archivos finales enriquecidos
    enriquecer_y_guardar("brain_CORE_1h.csv", "brain_FINAL_1h.csv")
    enriquecer_y_guardar("brain_CORE_1min.csv", "brain_FINAL_1min.csv")

📖 Leyendo brain_CORE_1h.csv...
🛠️ Añadiendo características de calendario...
💾 Guardando archivo final: brain_FINAL_1h.csv...
------------------------------
✅ ¡ARCHIVO FINAL LISTO!
Nuevas columnas añadidas: hour_sin, hour_cos, day_sin, day_cos, is_weekend
Total de columnas: 89
------------------------------
📖 Leyendo brain_CORE_1min.csv...
🛠️ Añadiendo características de calendario...
💾 Guardando archivo final: brain_FINAL_1min.csv...
------------------------------
✅ ¡ARCHIVO FINAL LISTO!
Nuevas columnas añadidas: hour_sin, hour_cos, day_sin, day_cos, is_weekend
Total de columnas: 89
------------------------------


In [1]:
import pandas as pd
import numpy as np

def autopsia_csv(fichero):
    print(f"\n--- Autopsia de {fichero} ---")
    df = pd.read_csv(fichero)
    
    # 1. Buscar ceros reales
    total_ceros = (df == 0).sum().sum()
    print(f"Empty/Zero values: {total_ceros} ({total_ceros/df.size:.2%} del total)")
    
    # 2. Buscar outliers extremos (posibles errores de lectura)
    # Si el valor máximo es 1000 veces mayor que la mediana, hay algo raro.
    for col in df.select_dtypes(include=[np.number]).columns[:5]: # Solo las primeras 5 para no saturar
        mediana = df[col].median()
        maximo = df[col].max()
        if mediana > 0 and (maximo / mediana) > 1000:
            print(f"⚠️ Columna {col}: Máximo ({maximo}) es >1000x la mediana ({mediana}). Posible outlier.")

    # 3. Buscar columnas con varianza cero (datos muertos)
    constantes = [col for col in df.columns if df[col].nunique() <= 1]
    print(f"Columns with constant values: {len(constantes)}")
    if constantes:
        print(f"Ejemplos de columnas muertas: {constantes[:3]}")

autopsia_csv("brain_CORE_1h.csv")
autopsia_csv("brain_CORE_1min.csv")


--- Autopsia de brain_CORE_1h.csv ---
Empty/Zero values: 162 (0.02% del total)
⚠️ Columna ADH_ADH: Máximo (117010326.0) es >1000x la mediana (12873.0). Posible outlier.
Columns with constant values: 0

--- Autopsia de brain_CORE_1min.csv ---
Empty/Zero values: 0 (0.00% del total)
⚠️ Columna ADH_ADH: Máximo (42692498.0) es >1000x la mediana (1263.0). Posible outlier.
Columns with constant values: 0


In [2]:
import os
import pandas as pd

# === CONFIGURACIÓN ===
ARCHIVO_1MIN = "brain_FINAL_1min.csv"
ARCHIVO_SALIDA = "brain_FINAL_1min_con_MACRO.csv"

# Ruta exacta donde están tus resultados de 1 hora
ROOT_MACRO = "./Results/results_Arq1_1h_W24"

def fusionar_predicciones_macro():
    print(f"📂 Cargando dataset principal: {ARCHIVO_1MIN}...")
    
    if not os.path.exists(ARCHIVO_1MIN):
        print(f"❌ Error: No se encontró {ARCHIVO_1MIN}")
        return

    df_1min = pd.read_csv(ARCHIVO_1MIN)
    df_1min['timestamp'] = pd.to_datetime(df_1min['timestamp'])

    # Extraer los nombres de los enlaces
    COLS_TIEMPO = ['hour_sin', 'hour_cos', 'day_sin', 'day_cos', 'is_weekend']
    enlaces = [c for c in df_1min.columns if c not in (['timestamp', 'hour', 'day_of_week'] + COLS_TIEMPO)]

    # 🔑 EL TRUCO DE ALINEACIÓN: Los minutos 15:00 a 15:59 se cruzan con la hora 16:00
    df_1min['llave_cruce'] = df_1min['timestamp'].dt.floor('h') + pd.Timedelta(hours=1)

    print(f"🔄 Pegando predicciones de 1h para {len(enlaces)} enlaces...")

    for idx, enlace in enumerate(enlaces, 1):
        ruta_macro = os.path.join(ROOT_MACRO, f"test_results_Arq1_GRU_1h_{enlace}.csv")
        nombre_columna_macro = f"MACRO_{enlace}"
        
        if not os.path.exists(ruta_macro):
            print(f"⚠️ [{idx}/{len(enlaces)}] Archivo no encontrado: {ruta_macro}. Rellenando con NaNs.")
            df_1min[nombre_columna_macro] = float('nan')
            continue
            
        # 1. Cargar el CSV de resultados de 1 hora de ese enlace
        df_macro = pd.read_csv(ruta_macro)
        df_macro['timestamp'] = pd.to_datetime(df_macro['timestamp'])
        
        # 2. Nos quedamos SOLO con la fecha y la predicción (renombrada)
        df_macro = df_macro[['timestamp', 'Prediccion_GRU']].rename(
            columns={'Prediccion_GRU': nombre_columna_macro}
        )
        
        # 3. Cruzar (Left Join) con el dataset de 1 minuto
        df_1min = pd.merge(
            left=df_1min,
            right=df_macro,
            left_on='llave_cruce',
            right_on='timestamp',
            how='left',
            suffixes=('', '_drop')
        )
        
        # 4. Limpiar columnas duplicadas que genera pandas al hacer merge
        if 'timestamp_drop' in df_1min.columns:
            df_1min.drop(columns=['timestamp_drop'], inplace=True)
            
        print(f"✅ [{idx}/{len(enlaces)}] Añadida columna: {nombre_columna_macro}")

    # Limpieza final: borramos la llave temporal de cruce
    df_1min.drop(columns=['llave_cruce'], inplace=True)

    # Guardar a disco
    print("\n" + "="*60)
    print(f"💾 Guardando archivo final: {ARCHIVO_SALIDA}...")
    df_1min.to_csv(ARCHIVO_SALIDA, index=False)
    print("🎉 ¡Proceso terminado con éxito!")
    print("="*60)

if __name__ == "__main__":
    fusionar_predicciones_macro()

📂 Cargando dataset principal: brain_FINAL_1min.csv...
🔄 Pegando predicciones de 1h para 81 enlaces...
✅ [1/81] Añadida columna: MACRO_ADH_ADH
✅ [2/81] Añadida columna: MACRO_ADH_CVK
✅ [3/81] Añadida columna: MACRO_ADH_HTW
✅ [4/81] Añadida columna: MACRO_ADH_HU
✅ [5/81] Añadida columna: MACRO_ADH_SPK
✅ [6/81] Añadida columna: MACRO_ADH_TU
✅ [7/81] Añadida columna: MACRO_ADH_UP
✅ [8/81] Añadida columna: MACRO_ADH_WIAS
✅ [9/81] Añadida columna: MACRO_ADH_ZIB
✅ [10/81] Añadida columna: MACRO_CVK_ADH
✅ [11/81] Añadida columna: MACRO_CVK_CVK
✅ [12/81] Añadida columna: MACRO_CVK_HTW
✅ [13/81] Añadida columna: MACRO_CVK_HU
✅ [14/81] Añadida columna: MACRO_CVK_SPK
✅ [15/81] Añadida columna: MACRO_CVK_TU
✅ [16/81] Añadida columna: MACRO_CVK_UP
✅ [17/81] Añadida columna: MACRO_CVK_WIAS
✅ [18/81] Añadida columna: MACRO_CVK_ZIB
✅ [19/81] Añadida columna: MACRO_HTW_ADH
✅ [20/81] Añadida columna: MACRO_HTW_CVK
✅ [21/81] Añadida columna: MACRO_HTW_HTW
✅ [22/81] Añadida columna: MACRO_HTW_HU
✅ [23/81] 